# Dijkstra's algorithm: watch the route costs change

A visual companion to the [Dijkstra lesson](../algorithms/network/dijkstra.md). Step through the priority queue, watch tentative distances improve and see why the first route found is not always the cheapest.

Open in [Google Colab](https://colab.research.google.com/github/geo-dev-hub/spatial-analysis/blob/main/notebooks/dijkstra-lab.ipynb) or run in JupyterLab with `notebooks/requirements.txt`.

The graph is a small directed teaching example. Edge weights are costs, not geographic distances.

In [ ]:
from pathlib import Path
import sys
from heapq import heappop, heappush
from itertools import count
from math import inf

import matplotlib.pyplot as plt
from ipywidgets import IntSlider, interact

current = Path.cwd().resolve()
repo_root = next((p for p in (current, *current.parents)
                  if (p / 'implementations/python/dijkstra').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Run this notebook from a checkout of geo-dev-hub/spatial-analysis.')
sys.path.insert(0, str(repo_root / 'implementations/python/dijkstra'))
from dijkstra import dijkstra

print('Repository:', repo_root)

## 1. Describe the network

Each directed edge has a cost. The algorithm adds these costs along a path. It does not know whether a cost means metres, minutes, money or something else; that meaning comes from the data and model.

The graph below has more than one route from S to T. We will find the route with the lowest total cost.

In [ ]:
graph = {
    'S': [('A', 4), ('B', 2)],
    'B': [('A', 1), ('C', 5), ('T', 9)],
    'A': [('C', 3), ('T', 7)],
    'C': [('T', 1)],
    'T': [],
}
positions = {'S': (0, 1), 'B': (1, 0), 'A': (2, 1), 'C': (3, 0), 'T': (4, 1)}

def draw_network(ax, distances=None, current_node=None, finalised=None):
    distances = distances or {}
    finalised = set(finalised or [])
    for node, edges in graph.items():
        x1, y1 = positions[node]
        for neighbour, weight in edges:
            x2, y2 = positions[neighbour]
            ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                        arrowprops={'arrowstyle': '->', 'shrinkA': 14, 'shrinkB': 14, 'lw': 1.7})
            ax.text((x1+x2)/2, (y1+y2)/2 + 0.10, str(weight), ha='center', va='center')
    for node, (x, y) in positions.items():
        ax.scatter([x], [y], s=900, marker='o')
        value = distances.get(node, inf)
        label = '∞' if value == inf else f'{value:g}'
        ax.text(x, y, f'{node}\n{label}', ha='center', va='center', color='white', weight='bold')
        if node in finalised:
            ax.scatter([x], [y], s=1150, marker='s', facecolors='none', linewidths=2)
    if current_node is not None:
        x, y = positions[current_node]
        ax.scatter([x], [y], s=1150, marker='*', facecolors='none', linewidths=2)
    ax.set_xlim(-0.6, 4.6); ax.set_ylim(-0.7, 1.7)
    ax.set_aspect('equal', adjustable='box'); ax.axis('off')

fig, ax = plt.subplots(figsize=(9, 3.8))
draw_network(ax)
ax.set_title('Directed network: labels on edges are costs')
plt.show()

## 2. Record each step

We use a priority queue: the next node removed is the one with the smallest tentative distance. When a shorter route is discovered, the distance and predecessor are updated and a new queue entry is added.

Older entries can remain in the heap. If an entry no longer matches the best-known distance, it is stale and can be skipped.

In [ ]:
def trace_dijkstra(graph, source):
    vertices = set(graph)
    for edges in graph.values():
        vertices.update(neighbour for neighbour, _ in edges)
    distances = {node: inf for node in vertices}
    predecessors = {node: None for node in vertices}
    distances[source] = 0.0
    sequence = count()
    queue = [(0.0, next(sequence), source)]
    finalised = []
    events = [{'kind': 'start', 'node': None, 'distances': distances.copy(),
               'predecessors': predecessors.copy(), 'finalised': [], 'updates': [], 'queue': queue.copy()}]
    while queue:
        current_distance, _, node = heappop(queue)
        if current_distance > distances[node]:
            events.append({'kind': 'stale', 'node': node, 'distances': distances.copy(),
                           'predecessors': predecessors.copy(), 'finalised': finalised.copy(),
                           'updates': [], 'queue': queue.copy()})
            continue
        finalised.append(node)
        updates = []
        for neighbour, weight in graph.get(node, []):
            candidate = current_distance + weight
            if candidate < distances[neighbour]:
                old = distances[neighbour]
                distances[neighbour] = candidate
                predecessors[neighbour] = node
                heappush(queue, (candidate, next(sequence), neighbour))
                updates.append((neighbour, old, candidate))
        events.append({'kind': 'settled', 'node': node, 'distances': distances.copy(),
                       'predecessors': predecessors.copy(), 'finalised': finalised.copy(),
                       'updates': updates, 'queue': queue.copy()})
    return events

events = trace_dijkstra(graph, 'S')
result = dijkstra(graph, 'S')
assert events[-1]['distances'] == result.distances
print('Queue events:', len(events) - 1)
print('Shortest route to T:', result.path_to('T'))
print('Total cost:', result.distances['T'])

## 3. Step through the priority queue

Move the slider to inspect the state after each queue removal. A star marks the node being processed, and a square outline marks nodes whose shortest distance is final. The distance shown inside each node is its best-known cost from S. A stale entry is skipped because a better value has already been found.

In [ ]:
def show_step(step=0):
    event = events[step]
    fig, ax = plt.subplots(figsize=(9, 4))
    draw_network(ax, event['distances'], event['node'], event['finalised'])
    ax.set_title(f"Step {step}: {event['kind']}" + (f" node {event['node']}" if event['node'] is not None else ''))
    plt.show()
    print('Finalised nodes:', event['finalised'])
    print('Distance updates:', event['updates'])
    print('Predecessors:', event['predecessors'])
    print('Pending queue entries:', sorted((round(item[0], 3), item[2]) for item in event['queue']))

interact(show_step, step=IntSlider(value=0, min=0, max=len(events)-1, step=1, description='Queue step'))

## 4. Reconstruct the route

The predecessor table records the node immediately before each destination on the best-known route. Follow those links backwards from T, then reverse the sequence.

Try changing an edge cost in the graph cell and rerunning the notebook. Which nodes are settled in a different order? Does the best route to T change?

In [ ]:
route = result.path_to('T')
print('Route:', ' → '.join(route))
print('Cost:', result.distances['T'])

fig, ax = plt.subplots(figsize=(9, 3.8))
draw_network(ax, result.distances)
for start, end in zip(route, route[1:]):
    x1, y1 = positions[start]; x2, y2 = positions[end]
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops={'arrowstyle': '->', 'shrinkA': 14, 'shrinkB': 14, 'lw': 4})
ax.set_title('Least-cost route from S to T')
plt.show()

## Experiments

1. Change the cost of S→A from 4 to 1. What changes in the final route?
2. Add a zero-cost edge. Why is zero allowed?
3. Add an isolated node. What distance should it receive?
4. Try a negative edge. The implementation should reject it. Explain why the greedy proof depends on non-negative weights.
5. Replace distance with travel time. What assumptions about traffic or service frequency are hidden in a static edge weight?

## Continue learning

- [Dijkstra lesson](../algorithms/network/dijkstra.md)
- [Python implementation](../implementations/python/dijkstra/README.md)
- [Step-by-step implementation walkthrough](../implementations/python/dijkstra/implementation-walkthrough.md)
- [Spatial indexing and query processing](../learning-paths/spatial-indexing.md)
